In [10]:
import sys
sys.path.append('C:\\Users\\yuyaa.LAPTOP-UEQT9ACJ\\Box\\Tokyo_Jobs\\Data_Collection\\Split_Page')

In [11]:
import os,cv2
os.chdir('C:\\Users\\yuyaa.LAPTOP-UEQT9ACJ\\Box\\Tokyo_Jobs\\Data_Collection\\Extract_Data')

from Extract import DetectCols, ToDict, AssignIndex
from Split import HoriPart, VertPart
from Read import Vision
from google.cloud import vision
from google.cloud.vision_v1 import types

In [12]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'C:\\Users\\yuyaa.LAPTOP-UEQT9ACJ\\Google ドライブ\\999. OCR\\Vision API Project-5912afb0bd62.json'
# Instantiates a client
client = vision.ImageAnnotatorClient()

In [13]:
import sys, json, os, cv2
import pandas as pd

Year,Showa='1942','17'

df = pd.read_csv(r'C:/Users/yuyaa.LAPTOP-UEQT9ACJ/Box/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

origin='C:/Users/yuyaa.LAPTOP-UEQT9ACJ/Box/YUYA_researchnote/'

In [14]:
StepAError,StepBError,MainError=[],[],[]
Dict={}
PosiDict={}
for Page in range(10, 130, 1):
    #Step A
    try:        
        path='C:\\Users\\yuyaa.LAPTOP-UEQT9ACJ\\Box\\'
        file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/Line/Page'+'{:03d}'.format(Page)+'.jpg'
        img=cv2.imread(os.path.join(path,file_path))

        #Convert to book page
        BookPage=2*Page-14
        Rows=df[(df['Page']==BookPage)]
        NxRows=df[(df['Page']==BookPage+1)]
        if len(Rows)==0:
            print('No Office at Right Side. Page'+str(BookPage))
        if len(NxRows)==0:
            print('No Office at Left Side. Page'+str(BookPage+1))

        texts=Vision(img,'zh',client)
    except:
        StepAError.append(Page)
        
    #Step B
    try:        
        sys.path.append('C:\\Users\\yuyaa.LAPTOP-UEQT9ACJ\\Box\\Tokyo_Jobs\\Data_Collection\\Split_Page')
        from Split import HoriPart, VertPart
        sys.path.append('C:\\Users\\yuyaa.LAPTOP-UEQT9ACJ\\Box\\Tokyo_Jobs\\Data_Collection\\Split_Office')
        from Organize import Filter, ConvertDict
        file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/Line/Page'+'{:03d}'.format(Page)+'.jpg'
        path=os.path.join(origin,file_path)

        try:
            HoriPoint=HoriPart(path)[0]
        except:
            print('Failed detecting Horizontal Point')
            HoriPoint=600

        try:
            VertPoint=VertPart(path)[1]
        except:
            print('Failed detecting Vertical Point')
            VertPoint=300

        ##Right Page
        BoxR=Filter(BookPage,texts,HoriPoint)
        BoxR=ConvertDict(BoxR)

        ##Left Page
        BoxL=Filter(BookPage+1,texts,HoriPoint)
        BoxL=ConvertDict(BoxL)

        sys.path.append('C:\\Users\\yuyaa.LAPTOP-UEQT9ACJ\\Box\\Tokyo_Jobs\\Data_Collection\\Detect')
        from Organize import FilterBox
        from Detect import DetectOffice
        LocList=[]

        #RightPage
        OfficeList=Rows['Office'].tolist()
        for Office in OfficeList:
            LocList.append(DetectOffice(BoxR, Office,VertPoint))
            BoxR=FilterBox(BoxR,LocList,VertPoint)

        #LeftPage
        OfficeList=NxRows['Office'].tolist()
        for Office in OfficeList:
            LocList.append(DetectOffice(BoxL, Office,VertPoint))
            BoxL=FilterBox(BoxL,LocList,VertPoint)

        Dict[Page]=LocList
    except:
        StepBError.append(Page)
        continue
        
    # Main Code 
    #Split quardrant into offices and detect Positions
    sys.path.append('C:\\Users\\yuyaa.LAPTOP-UEQT9ACJ\\Box\\Tokyo_Jobs\\Data_Collection\\Split_Position')
    from OrganizePosi import Split, Crop
    from DetectPosi import DetectPosi

    CrossWalk=pd.read_csv("C:/Users/yuyaa.LAPTOP-UEQT9ACJ/Box/Tokyo_Jobs/Processed_Data/PositionCrosswalk.csv")['Japanese']
    Positions=CrossWalk.tolist()[:-1]
    PosiDict['Pre'+str(Page)]=[]

    DF=Crop(texts,HoriPoint,VertPoint,Dict,Page)

    ##UR
    BoxList=Split(DF['UR']['Box'],DF['UR']['Thres'])
    PosiDict=DetectPosi(BoxList,DF['UR']['Thres'],PosiDict,Positions)

    ##LR
    BoxList=Split(DF['LR']['Box'],DF['LR']['Thres'])
    PosiDict=DetectPosi(BoxList,DF['LR']['Thres'],PosiDict,Positions)

    ##UL
    BoxList=Split(DF['UL']['Box'][1:],DF['UL']['Thres'])
    PosiDict=DetectPosi(BoxList,DF['UL']['Thres'],PosiDict,Positions)

    #LL
    BoxList=Split(DF['LL']['Box'],DF['LL']['Thres'])
    PosiDict=DetectPosi(BoxList,DF['LL']['Thres'],PosiDict,Positions)


No Office at Right Side. Page8
No Office at Right Side. Page10
Failed detecting Horizontal Point
No Office at Left Side. Page15
Failed detecting Horizontal Point
No Office at Left Side. Page19
Failed detecting Horizontal Point
Failed detecting Horizontal Point
No Office at Right Side. Page22
No Office at Left Side. Page23
Failed detecting Horizontal Point
Failed detecting Horizontal Point
No Office at Right Side. Page36
No Office at Right Side. Page38
No Office at Left Side. Page39
Failed detecting Horizontal Point
No Office at Right Side. Page40
No Office at Left Side. Page41
No Office at Right Side. Page44
No Office at Left Side. Page45
No Office at Right Side. Page46
No Office at Left Side. Page47
No Office at Left Side. Page49
Failed detecting Horizontal Point
Failed detecting Horizontal Point
No Office at Right Side. Page54
Failed detecting Horizontal Point
Failed detecting Horizontal Point
No Office at Left Side. Page85
No Office at Left Side. Page87
Failed detecting Horizontal P

# Summary of performance

**1. Non-Running Error**

In [15]:
from Show import Show
def ErrorRate(ErrorList,PageList):
    return(round(len(ErrorList)/len(range(10, 130, 1)),2))

In [16]:
ErrorRate(StepAError,range(10, 130, 1)),ErrorRate(StepBError,range(10, 130, 1)),ErrorRate(MainError,range(10, 130, 1))

(0.0, 0.16, 0.0)

In [20]:
StepBError

[13, 19, 22, 33, 39, 40, 41, 42, 44, 52, 53, 67, 81, 82, 86, 88, 89, 90, 121]

**2.Miss Assignment Error**

In [9]:
from Show import Show
def Show(PosiDict,Office,img,VertPoint,HoriPoint):
    copy=img.copy()
    MaxPage=len(PosiDict[Office])
    for Page in range(MaxPage):
        PosiList=[d['Position'] for d in PosiDict[Office][Page]]
        cv2.line(copy,(HoriPoint,0),(HoriPoint,800),(0,0,0),2)
        for n in range(len(PosiList)):
            XAxis,YAxis=PosiDict[Office][Page][n]['Location'][0],PosiDict[Office][Page][n]['Location'][1]
            if PosiDict[Office][Page][n]['Location'][1]<VertPoint:                   
                cv2.line(copy,(XAxis,0),(XAxis,VertPoint),(0,0,0),2)
            else:
                cv2.line(copy,(XAxis,VertPoint),(XAxis,800),(0,0,0),2)
    cv2.imshow('',copy)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [10]:
for Office in list(PosiDict.keys()):
    print(Office)
    if 'Pre' in Office:
        continue
    try:
        BookPage=df[df['Office']==Office]['Page'].tolist()[0]
    except:
        continue
    
    if BookPage%2==0:
        Page=(BookPage+14)//2
    else:
        Page=(BookPage+13)//2

    file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/Line/Page'+'{:03d}'.format(Page)+'.jpg'
    path=os.path.join(origin,file_path)
    
    try:
        HoriPoint=HoriPart(path)[0]
    except:
        print('Failed detecting Horizontal Point')
        HoriPoint=600
    try:
        VertPoint=VertPart(path)[1]
    except:
        print('Failed detecting Vertical Point')
        VertPoint=300
    
    img=cv2.imread(path)
    
    Show(PosiDict,Office,img,VertPoint,HoriPoint)


Pre10
議案課
区政課
報道課
統計課
Pre15
用品課
Failed detecting Horizontal Point
購買課
Failed detecting Horizontal Point
Pre20
動員課
貯蓄奨励課
商工課
Pre25
體力課
衛生課
Pre30
Pre35
大久保病院
Failed detecting Horizontal Point
大塚病院
Failed detecting Horizontal Point
Pre45
庶務課
計書課
Pre50
治水課
Failed detecting Horizontal Point
Pre55
葛飾区出張所
Failed detecting Horizontal Point
江戸川区出張所
Failed detecting Horizontal Point
築港課
Failed detecting Horizontal Point
Pre60
Pre65
Pre70
調査課
教習所
Pre75
車輛工場
三田電車営業所
Pre80
荒川電車営業所
城東電車営業所
濱松町自動車営業所
目黒自動車営業所
Pre85
研究課
監理課
作業課
Pre95
麻布区役所
赤坂区役所
Pre100
浅草区役所
Failed detecting Horizontal Point
Pre105
荏原区役所
Pre110
淀橋区役所
Pre115
荒川区役所


- Cause of Error
1. Horizontal Line inaccuracy: 4
2. Position not showing : 31
3. Inaccuracy in Office: 2| 庶務課，計書課, 調査課


In [11]:
CrossWalk=pd.read_csv("C:/Users/yuyaa.LAPTOP-UEQT9ACJ/Box/Tokyo_Jobs/Processed_Data/PositionCrosswalk.csv")['Japanese']
Positions=CrossWalk.tolist()[:-1]
Positions        

['主事',
 '技師',
 '技師',
 '投手',
 '技手',
 '手',
 '事務員',
 '嘱託員',
 '書記',
 '収納書記',
 '社会書記',
 '臨時',
 '視学',
 '守衛長',
 '運轉手',
 '運輸手',
 '監督',
 '保健婦',
 '醫員',
 '警員',
 '調薬員',
 '監視',
 '講師',
 '鑑定員',
 '薬剤員',
 '看護婦長',
 '清掃監督',
 '船長',
 '機関手',
 '骨豊力指導員']